# Algorithms for massive datasets project

## 1) Data loading

In [5]:
import os
import zipfile
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
with zipfile.ZipFile("imdb-dataset-of-top-1000-movies-and-tv-shows.zip", "r") as zip_ref:
    zip_ref.extractall("imdb_data")

HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDatasetMetadata (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001EBD5AE22D0>: Failed to resolve 'api.kaggle.com' ([Errno 11001] getaddrinfo failed)"))


In [6]:
import pandas as pd
from pyspark.sql import SparkSession

In [7]:
spark = (
    SparkSession.builder.appName("IMDB MBA").getOrCreate()
    
)
sc = spark.sparkContext

In [8]:
file_path = r"C:\Users\Alberto\Desktop\AMD\imdb_data\imdb_top_1000.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)
df.head(2)

[Row(Poster_Link='https://m.media-amazon.com/images/M/MV5BMDFkYTc0MGEtZmNhMC00ZDIzLWFmNTEtODM1ZmRlYWMwMWFmXkEyXkFqcGdeQXVyMTMxODk2OTU@._V1_UX67_CR0,0,67,98_AL_.jpg', Series_Title='The Shawshank Redemption', Released_Year='1994', Certificate='A', Runtime='142 min', Genre='Drama', IMDB_Rating=9.3, Overview='Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.', Meta_score='80', Director='Frank Darabont', Star1='Tim Robbins', Star2='Morgan Freeman', Star3='Bob Gunton', Star4='William Sadler', No_of_Votes='2343110', Gross='28,341,469'),
 Row(Poster_Link='https://m.media-amazon.com/images/M/MV5BM2MyNjYxNmUtYTAwNi00MTYxLWJmNWYtYzZlODY3ZTk3OTFlXkEyXkFqcGdeQXVyNzkwMjQ5NzM@._V1_UY98_CR1,0,67,98_AL_.jpg', Series_Title='The Godfather', Released_Year='1972', Certificate='A', Runtime='175 min', Genre='Crime, Drama', IMDB_Rating=9.2, Overview="An organized crime dynasty's aging patriarch transfers control of his clandestine empire to 

In [9]:
actors = ["Star1", "Star2", "Star3", "Star4"]
baskets = (
    df.select(actors).rdd.map(lambda row: [actor.strip()
          for actor in row
          if actor is not None and actor.strip() != ""
      ])
      .filter(lambda basket: len(basket) >= 2).map(lambda basket: sorted(set(basket)))
)

In [11]:
print(baskets.take(2))

[['Bob Gunton', 'Morgan Freeman', 'Tim Robbins', 'William Sadler'], ['Al Pacino', 'Diane Keaton', 'James Caan', 'Marlon Brando']]
